In [ ]:
import numpy as np
import eucare as ec
from eucare.base import angle_to_axis
import networkx as nx
import itertools

In [ ]:
eps = 0.1

# 3.1 Stage 1: Polygon Placement
def seed_polygon(n):
    # see Figure 7 in [1]
    G = ec.half.EuclideanPositionHEG(other=ec.prototiles.RegularEuclideanTile(n).make_graph(add_positions=True)[0])
    if n != 3:
        return G
    else:
        pos, _ = G.get_position_view()
        pos *= -1
        pos -= pos.min(0, keepdims=True)
        return G

def starting_border(G, seed_face):    
    try:
        if seed_face.order() != 3:
            h = next(h.rev for h in seed_face.halfedge_iter()
                     if h.rev.on_border() and max(h.orig['pos'][0], h.dest['pos'][0]) < -eps)
        else:
            h = next(h.rev for h in seed_face.halfedge_iter()
                     if h.rev.on_border() and max(h.orig['pos'][0], h.dest['pos'][0]) < eps)
    except StopIteration:
        raise NotImplemetedError('One of the seed faces edges must lie on the border')
    
    h0 = h
    while max(h.orig['pos'][0], h.dest['pos'][0]) < eps:
        h = h.nex
        assert h is not h0, f'No border edge in the positive quadrant!'
    return h

# e.g. code = '6-3-3'
def polygon_placement(code):
    code = code.replace(' ', '')  # delete whitespace
    phases = [[int(n) for n in c.split(',')] for c in code.split('-')]
    # Phase 1 (seed polygon phase)
    assert len(phases[0]) == 1, f'seed polygon phase should consist of one polygon only, got {len(phases[0])} ({phases[0]})'
    G = seed_polygon(phases[0][0])
    seed_face = next(iter(G.faces))
    
    # Other Phases
    # only use edges added in the last phase
    for phase in phases[1:]:
        for h in (h for h in G.halfedges if h.on_border()):
            h['old'] = h.attributes.get('old', 0) + 1
        attatch_at_list = [starting_border(G, seed_face)]
        while True:
            attatch_at_list.append(attatch_at_list[-1].nex)
            if attatch_at_list[-1] == attatch_at_list[0]:
                break
        attatch_at_list = attatch_at_list[:-1]
        attatch_at_list = [h for h in attatch_at_list if h.attributes.get('old', 0) <= 1]
        polys_to_attatch = [seed_polygon(n) if n > 0 else None for n in phase]
        i = 0
        for poly in polys_to_attatch:
            try:
                while not (attatch_at_list[i].on_border() and attatch_at_list[i] in G.halfedges):
                    i += 1
                attatch_at = attatch_at_list[i]
            except IndexError:
                raise IndexError(f'Not enough new edges to attatch polygons {phase} (only {len(attatch_at_list)} attatchment points available).')
            i += 1
            if poly is None:
                continue
            G.glue_graph_e2e(poly, attatch_at, next(h for h in poly.halfedges if h.on_border()))
    return G

        

polygon_placement('4-3-3,4').show()
# polygon_placement('4-3-3,4').show()
# polygon_placement('12-3,4-3,3').show()
polygon_placement('4-4,4-3,4-6').show()

In [ ]:
class GJHTile:
    def __init__(self, positions):
        positions = np.array(positions)
        assert len(positions.shape) == 2 and positions.shape[-1] == 2, f'{positions.shape}'
        self.positions = positions
        
    def order(self):
        return len(self.positions)
    
    def center(self):
        return self.positions.mean(0)
    
    def line_segment_halfway_points(self):
        pos1 = self.positions
        pos2 = np.concatenate([pos1[1:], pos1[:1]])
        halfway_points = (pos1 + pos2) / 2
        return halfway_points
    
    def line_segments(self):
        pos1 = self.positions
        pos2 = np.concatenate([pos1[1:], pos1[:1]])
        return np.stack([pos1, pos2], axis=-2)
    
    def copy(self):
        return GJHTile(self.positions.copy())
    
    def transform(self, mat):
        self.positions = apply_affine(mat, self.positions)
        return self
    
    def __repr__(self):
        return f'<{type(self).__name__}: order={self.order()}, positions={self.positions.round(2)}>'
    
    @classmethod
    def from_face(cls, f):
        assert isinstance(f, ec.half.Face), f'{type(f)}'
        return cls(positions=[v['pos'] for v in f.vertex_iter()])
        
class RegularGJHTile(GJHTile):
    def __init__(self, n):
        angles = np.linspace(0, 2*np.pi, n, endpoint=False) + np.pi / n
        radius = 1 / (2 * np.sin(np.pi / n))
        super().__init__(radius * np.stack([np.sin(angles), np.cos(angles)], -1))


In [ ]:
def polygon_graph(positions):
    vs = [ec.half.Vertex() for _ in range(len(positions))]
    G = ec.half.CyclicHalfedgeGraph(vs)
    for v, p in zip(vs, positions):
        v['pos'] = p
    G = ec.half.EuclideanPositionHEG(other=G)
    return G

def tiles_to_graph(tiles): 
#     tiles = [t if ec.base.signed_area(t.positions) > 0 else t[::-1] for t in tiles]
    Gs = [polygon_graph(t.positions if ec.base.signed_area(t.positions) > 0 else t.positions[::-1])
          for t in tiles]
    G = Gs[0]
    for G2 in Gs[1:]:
        G.add_graph(G2)
    G = remove_duplicates(G, eps=1e-1)
    return G
    
def show_tiles(tiles):    
    tiles_to_graph(tiles).show()
    
# show_tiles(tiles)

In [ ]:
from eucare.base import angle_to_axis
from eucare.overlap import group_closeby
from eucare.conversions import EHEG_from_nx

def U(alpha):
    return np.stack([np.sin(alpha), np.cos(alpha)])

def translation_mat(t):
    # moves to origin to t
    m = np.eye(3)
    m[:2, 2] = t
    return m

def rotation_mat(alpha):
    s, c = np.sin(alpha), np.cos(alpha)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def mirror_mat_line(line):
    # translate to origin
    t1 = translation_mat(-line[0])
    t2 = translation_mat(line[0])
    
    # rotate second point to x-axis
    angle = angle_to_axis(line[1] - line[0])
    r1 = rotation_mat(-angle)
    r2 = rotation_mat(angle)
    
    # mirror along x-axis
    mx = np.array([[1, 0, 0], [0, -1, 0], [0, 0, 1]], dtype=np.float64)
    
    # put it together
    return t2 @ r2 @ mx @ r1 @ t1

def mirror_mat_point(point):
    return mirror_mat_line(np.stack([point, point + np.array([point[1], -point[0]])]))

def rotation_mat_point(point, angle=np.pi):
    t1 = translation_mat(-point)
    t2 = translation_mat(point)
    r = rotation_mat(angle)
    return t2 @ r @ t1

def apply_affine(m, v):
    return np.moveaxis(m @ np.moveaxis(np.concatenate([v, np.ones_like(v[..., :1])], axis=-1), -1, -2), -1, -2)[..., :2]

def order_points(points):
    eps_angle = np.pi/1800
    angles = (-angle_to_axis(points) + np.pi/2 + 2*np.pi + eps_angle) % (2*np.pi) - eps_angle
    norms = np.linalg.norm(points, axis=-1)
    is_origin = norms < 0.1
    normed_points = points / np.clip(norms[:, None], a_min=1e-6, a_max=np.inf)
    angle_groups = group_closeby(normed_points, eps=eps_angle * 2 * np.pi)
    _, index, inverse = np.unique(angle_groups, return_index=True, return_inverse=True)
    angles = angles[index[inverse]]
    # origin last, it should never be selected
    ind = np.lexsort((norms, angles, is_origin))
    return ind

def remove_duplicates(G, eps=1e-6, exclude_edges=()):
    vs = list(G.vertices)
    pos = np.stack([v['pos'] for v in vs])
    
    groups = group_closeby(pos, eps)
    _, index, inverse = np.unique(groups, return_index=True, return_inverse=True)
    node_mapping = {i:j for i,j in enumerate(index[inverse])} 
    v_index = {v: node_mapping[i] for i, v in enumerate(vs)}

    nxG = nx.Graph()
    nxG.add_nodes_from(v_index.values())
    nx_positions = {i: pos[i] for i in node_mapping.values()}
    nxG.add_edges_from([(v_index[e.orig], v_index[e.dest]) for e in G.halfedges_representing_edges() 
                        if e not in set(exclude_edges).union({e.rev for e in exclude_edges})])

    G2 = EHEG_from_nx(nxG, nx_positions)
    G2.recompute_lengths_and_angles()
    return G2

# Stage 2 and following
def parse_transform(G, code):
#     tiles = [GJHTile.from_face(f) for f in G.faces]
    # code could e.g. be m30, r30, r(c2), r(v1)
    # first, find the origin
    code = code.split('(')
    mode = code[0][0]
    assert mode in {'r', 'm'}, f"Transformation type must be in {{'r', 'm'}}. Got '{mode}'."
    if code[0][1:]:
        angle = np.pi / 180 * int(code[0][1:])
    else:
        angle = None
        
    if len(code) == 1: # center is origin
        angle = np.pi if angle is None else angle
        assert angle > 0, f'Invalid angle: {angle}'
        angles = [angle]
        while angles[-1] is not None and 2 * angles[-1] < 2*np.pi:
            angles.append(angles[-1] * 2)
        if mode == 'm':
            return [mirror_mat_line([np.zeros(2), U(angle)]) for angle in angles]
        if mode == 'r':
            return [rotation_mat(angle) for angle in angles]
    else:
        assert angle is None, f"Either specify angle or origin! Got '{'('.join(code)}'."
        transform_code, origin_code = code
        assert origin_code[-1] == ')', f'{origin_code}'
        origin_code = origin_code[:-1]
        origin_type, index = origin_code[0], origin_code[1:]
        index = int(index) - 1
        if origin_type == 'c': # face center
            points = np.stack([f.midpoint() for f in G.faces])
            ind = order_points(points)
            point = points[ind[index]]
            if mode == 'm':
                return [mirror_mat_point(point)]
            else:
                return [rotation_mat_point(point)]
        elif origin_type == 'v':
            points = np.stack([v['pos'] for v in G.vertices])
            ind = order_points(points)
            point = points[ind[index]]
            if mode == 'm':
                return [mirror_mat_point(point)]
            else:
                return [rotation_mat_point(point)]
        elif origin_type == 'h':
            sides = np.stack([np.stack([h.orig['pos'], h.dest['pos']]) 
                      for h in G.halfedges_representing_edges()])
            points = sides.mean(1)
            ind = order_points(points)
            point = points[ind[index]]
            if mode == 'm':
                return [mirror_mat_line(sides[ind[index]])]
            else:
                return [rotation_mat_point(point)]
        else:
            assert False, f"Origin type must be in {{'c', 'v', 'h'}}. Got '{origin_type}'."
            

def add_transformed_tiles(tiles, mat, center_filter=None):
    center_filter = (lambda c: True) if center_filter is None else center_filter
    centers = np.stack([t.center() for t in tiles])
    transformed_centers = apply_affine(mat, centers)
    all_centers = np.concatenate([centers, transformed_centers])
    groups = group_closeby(all_centers, 0.01)
    _, index, inverse = np.unique(groups, return_index=True, return_inverse=True)
    new_ind = index[inverse] # first exemplar from each group
    new_ind = new_ind[new_ind >= len(tiles)] - len(tiles)
    return tiles + [tiles[i].copy().transform(mat) for i in new_ind 
                   if center_filter(transformed_centers[i])]

def make_gjh_tiling(code, bbox_size=20):
    code = code.replace(' ', '')
    stages = code.split('/')
    G = polygon_placement(stages[0])    
    
    tiles = [GJHTile.from_face(f) for f in G.faces]
    
    mats = []
    for stage in stages[1:]:
        ms = parse_transform(G, stage)
        for m in ms:
            tiles = add_transformed_tiles(tiles, m)
        mats.extend(ms)
        if len(tiles) > len(G.faces):
            G = tiles_to_graph(tiles)
    
    for i in itertools.count():
        n_tiles_before = len(tiles)
        for m in mats:
            tiles = add_transformed_tiles(tiles, m, center_filter=lambda c: np.max(np.abs(c)) < bbox_size/2)
        if len(tiles) == n_tiles_before or i > 1000:
            break
    
    return tiles_to_graph(tiles)
        

code = '4-6,4-0,3,3/m/r(v1)/r(h25)'
G = make_gjh_tiling(code)
print(code)
G.show()

In [ ]:
s = """
REGULAR
3
6 3/m30/r(h2)
6
3 6/m30/r(h1)
4
4 4/m45/r(h1)
UNIFORM
3.122 12-3/m30/r(h3)
3.4.6.4 6-4-3/m30/r(c2)
4.6.12 12-6,4/m30/r(c2)
(3.6)2 6-3-6/m30/r(v4)
4.82 8-4/m90/r(h4)
3
2
.4.3.4 4-3-3,4/r90/r(h2)
3
3
.42 4-3/m90/r(h2)
3
4
.6 6-3-3/r60/r(h5)
2 UNIFORM
3
6
; 32
.4.3.4 3-4-3/m30/r(c3)
3.4.6.4; 32
.4.3.4 6-4-3,3/m30/r(h1)
3.4.6.5; 33
.42 6-4-3-3/m30/r(h5)
3.4.6.4; 3.42
.6 6-4-3,4-6/m30/r(c4)
4.6.12; 3.4.6.4 12-4,6-3/m30/r(c3)
3
6
; 32
.4.12 12-3,4-3/m30/r(c3)
3.122
; 3.4.3.12 12-0,3,3-0,4/m45/m(h1)
3
6
; 32
.62 3-6/m30/r(c2)
[36
; 34
.6]1 6-3,3-3/m30/r(h1)
[36
; 34
.6]2 6-3-3,3-3/r60/r(h8)
3
2
.62
; 34
.6 6-3/m90/r(h1)
3.6.3.6; 32
.62 6-3,6/m90/r(h3)
[3.42
.6; 3.6.3.6]1 6-3,4-6-3,4-6,4/m90/r(c6)
[3.42
.6; 3.6.3.6]2 6-3,4/m90/r(h4)
[33
.42
; 32
.4.3.4]1 4-3,3-4,3/r90/m(h3)
[33
.42
; 32
.4.3.4]2 4-3,3,3-4,3/r(c2)/r(h13)/r(h45)
[44
; 33
.42
]
1 4-3/m(h4)/m(h3)/r(h2)
[44
; 33
.42
]
2 4-4-3-3/m90/r(h3)
[36
; 33
.42
]
1 4-3,4-3,3/m90/r(h3)
[36
; 33
.42
]
2 4-3-3-3/m90/r(h7)/r(h5)
3-UNIFORM (2 VERTEX TYPES)
(3.4.6.4)2
; 3.42
.6 6-4-3,4-6,3/m30/r(c2)
[(36
)
2
; 34
.6]1 6-3-3/m30/r(v3)
[(36
)
2
; 34
.6]2 6-3-3-3-0,3/m30/r(v2)
[(36
)
2
; 34
.6]3 6-3-3,3-3-3-0,3/r60/r(h7)
3
6
; (34
.6)2 3-3,3-6/m90/r(h6)
3
6
; (32
.4.3.4)2 3-4-3,3/m30/m(h2)
(3.42
.6)2
; 3.6.3.6 4-6,4-4,3,3/m90/r(h4)
[3.42
.6; (3.6.3.6)2
]
1 4-6,4-0,3,3/m/r(v1)/r(h25)
[3.42
.6; (3.6.3.6)2
]
2 4-6,4-0,3,3/m90/r(v1)
3
2
.62
; (3.6.3.6)2 6-3,0,3,3,3,3/r(h4)/r(v15)/r(v30)
(34
.6)2
; 3.6.3.6 6-3,3-0,3/r/r(v1)/r(h12)
[33
.42
; (44
)
2
]
1 4-4-4-3/m90/r(h4)
[33
.42
; (44
)
2
]
2 4-4-3/r(h6)/m(h5)/r(h3)
[(33
.42
)
2
; 44
]
1 4-4-3-3-4/m90/r(h10)/r(c3)
[(33
.42
)
2
; 44
]
2 4-3,4-3,3-4/m90/r(h3)
(33
.42
)
2
; 32
.4.3.4 4-4,3,4-3,3,3-3,4-3-4/r/r(h17)/r(h18)
3
3
.42
; (32
.4.3.4)2 4-3,3-0,4,3/r/r(h2)/r(h18)
[36
; (33
.42
)
2
]
1 4-3,0,3-3-3/r(h5)/r(h19)/m(h18)
[36
; (33
.42
)
2
]
2 4-3,0,3-3/r(h3)/r(h15)/m(h14)
[(36
)
2
; 33
.42
]
1 4-3-3-3-3-3/m90/r(h3)
[(36
)
2
; 33
.42
]
2 4-3-3-3-3/m90/r(h2)/m(h22)
3-UNIFORM (3 VERTEX TYPES)
3.42
.6; 3.6.3.6; 4.6.12 12-6,4-3,3,4/m30/r(c5)
3
6
; 32
.4.12; 4.6.12 12-3,4,6-3/m60/m(c5)
3
2
.4.12; 3.4.6.4; 3.122 6-4-3,12,3-3/m30/r(h2)
3.4.3.12; 3.4.6.4; 3.122 6-4-3,3-12-0,0,0,3/m30/r(c2)
3
3
.42
; 32
.4.12; 3.4.6.4 12-4,3-6,3-0,0,4/m30/r(h11)
3
6
; 33
.42
; 32
.4.12 12-3,4-3-3-3/m30/m(h9)
3
6
; 32
.4.3.4; 32
.4.12 12-3,4-3,3/m30/r(v1)
3
4
.6; 33
.42
; 32
.4.3.4 6-3-3-4-3,3/m30/r(h10)
3
6
; 32
.4.3.4; 3.42
.6 3-4-3,4-6/m30/r(c5)
3
6
; 33
.42
; 3.4.6.4 6-4-3,4-3,3/m30/r(c5)
3
6
; 32
.4.3.4; 3.4.6.4 6-4-3,3-4,3,3-3/r60/r(v5)
3
6
; 33
.42
; 32
.4.3.4 3-4-3-3/m30/r(h6)
3
2
.4.12; 3.4.3.12; 3.122 12-4-3,3/m90/r(h6)
3.4.6.4; 3.42
.6; 44 6-4,3-3,0,4-6/m90/r(v5)
3
2
.4.3.4; 3.4.6.4; 3.42
.6 6-4,3-3,3,4-0,0,6,3/m90/r(h17)/m(h1)
3
3
.42
; 32
.4.3.4; 44 4-3-3-0,4/r90/r(h3)
[3.42
.6; 3.6.3.6; 44
]
1 4-4-3,4-6/m/r(c3)/r(h29)
[3.42
.6; 3.6.3.6; 44
]
2 4-4,4-3,4-6/m90/r(c5)/r(v1)
[3.42
.6; 3.6.3.6; 44
]
3 6-3,4-0,4,4-0,4/m90/r(h9)
[3.42
.6; 3.6.3.6; 44
]
4 6-4,3,3-4/m(h4)/r/r(v15)
3
3
.42
; 32
.62
; 3.42
.6 4-6-3,0,3,3-0,0,4/m90/r(h4)
[32
.62
; 3.42
.6; 3.6.3.6]1 4-6,4,3-0,3,3,0,6/m(h2)/m
[32
.62
; 3.42
.6; 3.6.3.6]2 4-6,4-0,3,3/r(h2)/m90/r(c9)
3
4
.6; 33
.42
; 3.42
.6 4-6,4-0,3,3-0,3,3/r/r(c1)/r(h17)
[32
.62
; 3.6.3.6; 63
]
1 6-6-3,3,3/r60/r(h2)
[32
.62
; 3.6.3.6; 63
]
2 6-6,6,3-3,3/m//r(h8)/r(h49)
3
4
.6; 32
.62
; 63 6-3-3/m/r(h3)/r(h15)
3
6
; 32
.62
; 63 3-6/r60/m(c2)
[36
; 34
.6; 32
.62
]
1 6-3-3,3-3,3-0,3/r(h7)/r(h29)/r(h29)
[36
; 34
.6; 32
.62
]
2 3-3,6-3/m/r(h6)/r(c6)
[36
; 34
.6; 32
.62
]
3 6-3-3/m90/r(h2)
[36
; 34
.6; 3.6.3.6]1 3-3,3-3,6,3/m90/r(v1)/r(v15)
[36
; 34
.6; 3.6.3.6]2 3-3-6-0,3/r60/m(c1)
[36
; 34
.6; 3.6.3.6]3 3-3-6/r60/r(v4)
[36
; 33
.42
; 44
]
1 4-4-3-3/m90/r(h7)/r(v1)
[36
; 33
.42
; 44
]
2 4-4-3-3-3/m90/r(h9)/r(h3)
[36
; 33
.42
; 44
]
3 4-4-3-3-3/m(h9)/r(h1)/r(v1)
[36
; 33
.42
; 44
]
4 4-4-3-3-3/m(h9)/r(h1)/r(h3)
"""

In [ ]:
lines = s.split('\n')
codes = [l.split(' ')[-1] for l in lines if '/' in l]
codes = [c.replace('//', '/') for c in codes]
# codes = [c if c != '4-6,4,3-0,3,3,0,6/m(h2)/m' else '4-6,4,3-0,3,3,0,6/r(h4)/m(h6)/m(h2)/m' for c in codes]
codes = [c for c in codes if c != '4-6,4,3-0,3,3,0,6/m(h2)/m']

In [ ]:
from eucare.classifiers import congruency_classifier, AdjacencyClassifier

# G = make_gjh_tiling('4-3,3-4,3/r90/m(h3)')
G = make_gjh_tiling('6-3-3,3-3-3-0,3/r60/r(h7)')
plotting_kwargs=dict(render_faces=True, render_edges=False, render_vertices=False)
cc = congruency_classifier()
for f in G.faces:
    f['congruency_class'] = cc.classify(f)

ac = AdjacencyClassifier('congruency_class')
for f in G.faces:
    f['color_key'] = ac.classify(f)


G.show(**plotting_kwargs)

In [ ]:
# {3,6}
"""
a: a1 a1 a1
"""
# 3.6.3.6
"""
a: [b, b, b, b, b, b]
b: [a, a, a]
"""
# 3.3.3.4.4
"""
t: [s1, t2, t3]
s: [t1, s2, t1, s2]
"""

In [ ]:
from itertools import count

"""
1. classify faces by congruency, key 'f0'
2. classify edges by h.face['congruency_key'] and h.rev.face['f0'], assign 'h0'
3. classify faces with by cyclic classifier on edge keys 'h0', assign 'f1'
4. repeat 2. and 3. until the number of classes stagnates
5. select examplars fs of faces (inside graph), with 'any sides' hs
6. label edge classes by face they belong to, plus number along it starting from f.any_side (for f in fs)
7. generate final yaml code for pattern
8. generate tileset from yaml code

# TODO: plot tilesets (tiles in grid, text in / outside edges)
"""
from eucare.classifiers import *
from eucare.overlap import fast_group_closeby

class EdgeClassifier(RepresentationClassifier):
    def __init__(self, face_dict, edge_dict):
        super().__init__()
        self.face_dict = face_dict
        self.edge_dict = edge_dict
        
    def _compare_representations(self, query_rep, saved_rep):
        return np.all(query_rep == saved_rep)
        
    def _represent_item(self, h):
        return np.array([
            self.edge_dict.get(h, -1),
            self.edge_dict.get(h.rev, -1),
            self.edge_dict.get(h.nex, -1),
            self.edge_dict.get(h.pre, -1),
            self.face_dict.get(h.face, -1),
            self.face_dict.get(h.rev.face, -1),
        ], dtype=np.float32)
    
    def _get_index(self, item):
        query_rep = self._represent_query_item(item)
        if np.any(query_rep < 0):
            return -1
        if not self.represented_first and self.current_count == 1:  # compute representation that was skipped for performance (see below)
            self.count_to_repr[0] = self._represent_item(self.count_to_repr[0])
            self.represented_first = True
        for index, rep in self.count_to_repr.items():
            if self._compare_representations(query_rep, rep):
                return index
        if self.current_count == 0:  # for better performance, calculate representation only at first comparison
            self.count_to_repr[0] = item
        else:
            self.count_to_repr[self.current_count] = self._represent_item(item)
        self.current_count += 1
        return self.current_count - 1
    
class FaceClassifier(CyclicClassifier):
    def __init__(self, edge_dict):
        super(FaceClassifier, self).__init__(tolerance=0)
        self.edge_dict = edge_dict

    def _represent_query_item(self, f):
        return np.array([self.edge_dict[h] for h in f.halfedge_iter()])
    
    def _get_index(self, item):
        query_rep = self._represent_query_item(item)
        if np.any(query_rep < 0):
            return -1
        if not self.represented_first and self.current_count == 1:  # compute representation that was skipped for performance (see below)
            self.count_to_repr[0] = self._represent_item(self.count_to_repr[0])
            self.represented_first = True
        for index, rep in self.count_to_repr.items():
            if self._compare_representations(query_rep, rep):
                return index
        if self.current_count == 0:  # for better performance, calculate representation only at first comparison
            self.count_to_repr[0] = item
        else:
            self.count_to_repr[self.current_count] = self._represent_item(item)
        self.current_count += 1
        return self.current_count - 1

# G = make_gjh_tiling('4-3,3-4,3/r90/m(h3)')

def graph_to_tiling(G):

    # 1. classify faces by congruency, key 'f0'
    cc = congruency_classifier()
    fs = list(G.faces)
    face_dicts = [{f: cc.classify(f) for f in fs}]

    # 2.classify edges by length
    hs = list(G.halfedges)
    lengths = np.array([np.linalg.norm(h.orig['pos'] - h.dest['pos']) for h in hs])
    length_groups = fast_group_closeby(lengths[:, None], eps=1e-6)
    edge_dicts = [{h: lg for h, lg in zip(hs, length_groups)}]

    before = 0
    for _ in count():
        hc = EdgeClassifier(face_dicts[-1], edge_dicts[-1])
        edge_dicts.append({h: hc.classify(h) for h in hs})

        fc = FaceClassifier(edge_dicts[-1])
        face_dicts.append({f: fc.classify(f) for f in fs})

        after = len(set(edge_dicts[-1].values())) + len(set(face_dicts[-1].values()))
        if before >= after:
            break
        before = after

#     for f in G.faces:
#         face_class = face_dicts[-2][f]
#         f['color_key'] = face_class if face_class >=0 else (1, 1, 1)
#     G.show()

    #5. select examplars fs of faces (inside graph), with 'any sides' hs
    exemplar_fs = []
    for key in set(face_dicts[-2].values()):
        if key < 0:
            continue
        exemplar_fs.append(next(f for f in fs if face_dicts[-2][f] == key and face_dicts[-1][f] >= 0))

    exemplar_fs = sorted(exemplar_fs, key=lambda f: -f.order())
    tile_name_lookup = {f: chr(97 + i) for i, f in enumerate(exemplar_fs)}

    edge_name_lookup = dict()
    for f in exemplar_fs:
        edge_classes = {edge_dicts[-2][h] for h in f.halfedge_iter()}
        for i, h in enumerate(f.halfedge_iter()):
            edge_class = edge_dicts[-2][h]
            if edge_class not in edge_name_lookup:
                edge_name_lookup[edge_class] = f'{tile_name_lookup[f]}{i}'

    lines = '\n'.join([f'{tile_name_lookup[f]}: ' + str([edge_name_lookup[edge_dicts[-2][h.rev]] 
                                                  for h in f.halfedge_iter()]) 
                       for f in exemplar_fs]).replace("'", "")
#     for f in G.faces:
#         f['color_key'] = f in exemplar_fs
#     G.show()
    
    return lines

#6. label edge classes by face they belong to, plus number along it starting from f.any_side (for f in fs)
#7. generate final yaml code for pattern

In [ ]:
from eucare.prototiles import RegularEuclideanTile
from tqdm.auto import tqdm
import yaml

def load_tileset(tile_info):
    tile_info = yaml.load(tile_info, Loader=yaml.SafeLoader)
    tile_dict = {key: RegularEuclideanTile(
        n=len(sides), 
        edge_labels=[f'{key}{i}' for i in range(len(sides))],
        face_label=key
    )
                for key, sides in tile_info.items()}
    for tile_name, sides in tile_info.items():
        tile = tile_dict[tile_name]
        for inner, outer in zip(tile_dict[tile_name].edge_labels, sides):
            other_tile = tile_dict[outer[0]]
            tile.edge_instructions[inner] = other_tile.attach_instruction(outer)
    return list(tile_dict.values())[::-1]


# tiling_dict = {}
# problematic_codes = []
# for code in tqdm(codes):
#     try:
#         try:
#             G = make_gjh_tiling(code, bbox_size=20)
#             lines = graph_to_tiling(G)
#         except Exception as e:
#             print(f"Error: {e}. Trying again with larger graph")
#             G = make_gjh_tiling(code, bbox_size=40)
#             lines = graph_to_tiling(G)

#         lines = f'# GJH: {code}\n{lines}\n'
#         print(lines)
#         tiling_dict[code] = lines
# #         tiles = load_tileset(lines)
# #         G = ec.example_graphs.from_tiles(tiles, rings=5)
#     except Exception as e:
#         import traceback
#         print(traceback.format_exc())
#         problematic_codes.append(code)
        

In [ ]:
import os
from glob import glob

base_dir = 'archimedian'
tiling_dict = {}
for path in sorted(glob(os.path.join(base_dir, '*.yml'))):
    with open(path, 'r') as f:
        lines = f.read()
        
    gjh_code = (lines.split('\n')[0].split()[-1])
    tiling_dict[gjh_code] = lines 

In [ ]:
%pylab inline

from eucare.example_graphs import complete_vertex
from tqdm.auto import tqdm

def angle_profile(G):
    h0 = next(h for h in G.halfedges if h.on_border())
    hs = [h0.nex]
    while hs[-1] is not h0 and len(hs) < G.order:
        hs.append(hs[-1].nex)
    assert hs[-1] is h0, f'Border is not a cycle'
    angles = np.array([np.pi - 2*np.pi/len(hs) - h.orig.angle_sum() for h in hs])
    angles = np.cumsum(angles)
    angles -= np.mean(angles)
    angles = np.cumsum(angles)
    angles -= np.min(angles)
    return [h.dest for h in hs], angles
    
def next_vertices(G, domain=None, eps=1e-6):
    vs, angles = angle_profile(G)
    if domain is not None:
        mask = domain.contains(np.array([v['pos'] for v in vs]))
        vs, angles = np.array(vs)[mask], angles[mask]
    if len(vs) == 0:
        return []
    maximum = np.max(angles)
    return [vs[i] for i in np.argwhere(angles > maximum - eps)[:, 0]]
    
def add_vertex_ring(G, domain=None):
    vs = next_vertices(G, domain)
    for v in vs:
        h = v.any_outgoing
        complete_vertex(G, v)

class Domain():
    def contains(self, points):
        raise NotImplementedError
        
    def tile_with(self, tiles, max_steps=np.inf, offset=None):
        G = ec.example_graphs.from_tiles(tiles, rings=0)
        ps, vs = G.get_position_view()
        if offset is not None:
            ps[:] += offset[None]
        before = len(G.faces)
        for i in count():
            add_vertex_ring(G, domain=self)
            after = len(G.faces)
            if after == before: # domain is filled
                break 
            if i >= max_steps:
                break
            before = after
        return G
        
class SquareDomain(Domain):
    def __init__(self, sidelength):
        super().__init__()
        self.sidelength = sidelength
        
    def contains(self, points):
        return np.max(np.abs(points), axis=-1) < self.sidelength / 2
    
class RectangleDomain(Domain):
    def __init__(self, a, b, eps=1e-6):
        super().__init__()
        self.size = np.array([a, b]) + eps
        
    def contains(self, points):
        return np.max(np.abs(points / self.size[None]), axis=-1) < 0.5    


def render_tiling(tiles, sidelength=20, resolution=15, **render_kwargs):
    G = ec.example_graphs.from_tiles(tiles, rings=0)

    before = len(G.faces)
    for _ in range(1000):
        add_vertex_ring(G, domain=SquareDomain(sidelength * 1.1))
        after = len(G.faces)
        if after == before: # domain is filled
            break 
        before = after
    
    for f in G.faces:
        f['color_key'] = f['label']
    for h in G.halfedges:
        h['color_key'] = (0, 0, 0)
    G.show(
        scale=resolution, 
        height=sidelength * resolution, 
        render_vertices=False, 
        render_faces=True,
        face_inset=0,
        **render_kwargs
    )


In [ ]:
# how to find mirror symmetry in tiles?
# if symmetry exists, each tiles has an image under it, either itself or another tile.
# the mirror image has the same number of sides, and the same

In [ ]:
code = '12-3,3,4,6-4/m/r/r(h7)'
G = make_gjh_tiling(code, bbox_size=20)
lines = graph_to_tiling(G)
tiles = load_tileset(lines)
render_tiling(tiles, resolution=40)

In [ ]:
# code = '4-3-3,4/r90/r(h2)'
# code = '6/m30/r(h1)'
code = '3/m30/r(h2)'
tiles = load_tileset(tiling_dict[code])
domain = RectangleDomain(4.5, 21)
G = domain.tile_with(tiles)
G.show()

ps, vs = G.get_position_view()
k = ps.copy()
k = np.array([complex(*ki) for ki in k])
k -= np.mean(k)
k *= 1j
k += 3j - np.min(k.imag) * 1j
# k = k**1.2 # hex to pentagon
k = 1/k # hex to square
# k = k**3 # hex to triangle
# k = k**1.333333333333 # square to triangle

# k *= np.std(np.linalg.norm(ps, axis=-1)) / np.std(np.abs(k))

k = np.stack([k.real, k.imag], axis=-1)
ps[:] = k
G.show()

In [ ]:
def central_face(G):
    fs = list(G.faces)
    return fs[np.argmin([np.linalg.norm(f.midpoint()) for f in fs])]

code = '3/m30/r(h2)'
tiles = load_tileset(tiling_dict[code])
n = 20
m = 4
assert n > 3, m > 0
a = np.sqrt(3)/2
domain = RectangleDomain(a * (m+1), n+2)
G = domain.tile_with(tiles, offset=np.array([-a/2, 0.5]))
ps, vs = G.get_position_view()
mask = domain.contains(ps)
G.delete_subset(np.array(vs)[~mask])
# G.show()
ps, vs = G.get_position_view()

period = n

k = ps.copy()
k = np.array([complex(*ki) for ki in k])
k -= np.mean(k)
k -= np.min(k.real)
k = np.exp(k /period * 2 * np.pi)

k *= np.exp(2j * np.pi / n / 2)
k += 0.3
k = 1/k

k *= np.std(np.linalg.norm(ps, axis=1)) * np.std(np.abs(k))

k = np.stack([k.real, k.imag], axis=-1)
ps[:] = k

G = remove_duplicates(G, eps=1e-3)
G.delete_subset([f for f in G.faces if f.order() > 3])
# G.delete_subset(central_face(G))

G.show()

In [ ]:
from eucare.half import Vertex, HalfEdge, Face

def save_graph(G, attributes_to_save=('pos', 'length', 'in_angle', 'color_key')):
    # convert Graph to dict
    vertex_labels = {v: f'v{i}' for i, v in enumerate(G.vertices)}
    halfedge_labels = {h: f'h{i}' for i, h in enumerate(G.halfedges)}
    face_labels = {f: f'f{i}' for i, f in enumerate(G.faces)}
    
    labels = {None: None}
    labels.update(vertex_labels)
    labels.update(halfedge_labels)
    labels.update(face_labels)
    
    def represent_attributes(obj):
        result = {}
        for attr in attributes_to_save:
            if attr in obj.attributes:
                value = obj[attr]
                if isinstance(value, np.ndarray):
                    value = value.tolist()
                if np.isscalar(value):
                    value = float(value)
                result[attr] = value
        return result
    
    def add_attributes(func):
        def wrapped(obj):
            result = func(obj)
            attrs = represent_attributes(obj)
            if attrs:
                result['attributes'] = attrs
            return result
        return wrapped
    
    @add_attributes
    def represent_vertex(v):
        return dict(any_outgoing=labels[v.any_outgoing])
    
    @add_attributes
    def represent_halfedge(h):
        return dict(
            orig=labels[h.orig],
            dest=labels[h.dest],
            rev=labels[h.rev],
            nex=labels[h.nex],
            pre=labels[h.pre],
            face=labels[h.face]
        )
    
    @add_attributes
    def represent_face(f):
        return dict(any_side=labels[f.any_side])
    
    vertex_dict = {label: represent_vertex(v) for v, label in vertex_labels.items()}
    halfedge_dict = {label: represent_halfedge(h) for h, label in halfedge_labels.items()}
    face_dict = {label: represent_face(f) for f, label in face_labels.items()}
    
    graph_dict = dict(vertices=vertex_dict, halfedges=halfedge_dict, faces=face_dict)
    return graph_dict

def load_graph(graph_dict):
    def unwrap_attributes(obj_dict):
        result = {}
        for key, value in obj_dict.pop('attributes', {}).items():
            if isinstance(value, list):
                try:
                    value = np.array(value, dtype=np.float64)
                except Exception:
                    pass
            result[key] = value
        return result
    
    lookup = {None: None}
    # create the halfedges
    for label in graph_dict['halfedges']:
        lookup[label] = HalfEdge()
    
    vs = set()
    for label, v_dict in graph_dict['vertices'].items():
        attrs = unwrap_attributes(v_dict)
        v_dict['any_outgoing'] = lookup[v_dict['any_outgoing']]
        v = Vertex(**v_dict)
        v.attributes = attrs
        lookup[label] = v
        vs.add(v)
        
    fs = set()
    for label, f_dict in graph_dict['faces'].items():
        attrs = unwrap_attributes(f_dict)
        f_dict['any_side'] = lookup[f_dict['any_side']]
        f = Face(**f_dict)
        f.attributes = attrs
        lookup[label] = f
        fs.add(f)
    
    hs = set()
    for label, h_dict in graph_dict['halfedges'].items():
        attrs = unwrap_attributes(h_dict)
        h = lookup[label]
        for key in ['orig', 'dest', 'rev', 'nex', 'pre', 'face']:
            setattr(h, key, lookup[h_dict.pop(key, None)])
        h.attributes = attrs
        hs.add(h)
        
    G = ec.half.EuclideanPositionHEG()
    G.vertices=vs
    G.faces=fs
    G.halfedges=hs
    return G

G.recompute_lengths_and_angles()
%time graph_dict = save_graph(G)
%time load_graph(graph_dict)
G2.show()

graph_dict = save_graph(G)
print(yaml.dump(graph_dict))

In [ ]:
# from eucare.io import *

%time save_graph('test_save.heg', G, exist_ok=True)
%time G2 = load_graph('test_save.heg')
G2.show()

In [ ]:
next(v for v in G.vertices)['pos'].dtype

##### render all the tilings
for i, (code, lines) in enumerate(tqdm(tiling_dict.items())):
    print(code)
    tiles = load_tileset(lines)
    filename = os.path.join(base_dir, f"images/{str(i).zfill(2)}_{code.replace('/', '_')}")
    render_tiling(tiles, filename=filename)

In [ ]:
a = [np.mean(p) for p in angle_profiles]
plt.plot(a)

In [ ]:
'6/m30/r(h1)'
